# AI-Based Fraud Detection System
### Training Notebook — Aakriti Singh
---
This notebook walks through the full ML pipeline:
1. Load & explore data
2. Preprocess (scale, clean)
3. Handle class imbalance with SMOTE
4. Train Logistic Regression and Random Forest
5. Evaluate with Accuracy, Precision, Recall, F1, AUC-ROC
6. Save models

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
from preprocess import load_data

# Use the sample for demo; replace with full Kaggle dataset for real training
DATA_PATH = '../data/creditcard_sample.csv'
df = load_data(DATA_PATH)

print('Shape:', df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Class distribution
class_counts = df['Class'].value_counts()
print('Class distribution:')
print(f'  Legit (0): {class_counts[0]}')
print(f'  Fraud (1): {class_counts[1]}')
print(f'  Fraud %  : {class_counts[1]/len(df)*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
axes[0].bar(['Legit','Fraud'], class_counts.values, color=['#375623','#C00000'])
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Amount distribution by class
df[df['Class']==0]['Amount'].hist(ax=axes[1], bins=30, alpha=0.6, color='steelblue', label='Legit')
df[df['Class']==1]['Amount'].hist(ax=axes[1], bins=30, alpha=0.8, color='red',       label='Fraud')
axes[1].set_title('Transaction Amount Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Preprocess & SMOTE

In [ ]:
from preprocess import preprocess, apply_smote, get_train_test_split

X, y = preprocess(df, fit_scaler=True)
X_res, y_res = apply_smote(X, y)
X_train, X_test, y_train, y_test = get_train_test_split(X_res, y_res)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 4. Train Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib, os

os.makedirs('../models', exist_ok=True)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
joblib.dump(lr, '../models/logistic_regression.pkl')
print('Logistic Regression trained and saved ✓')

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
joblib.dump(rf, '../models/random_forest.pkl')
print('Random Forest trained and saved ✓')

## 5. Evaluate Models

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

for name, model in [('Logistic Regression', lr), ('Random Forest', rf)]:
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc     = roc_auc_score(y_test, y_proba)
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['Legit','Fraud']))
    print(f'AUC-ROC: {auc:.4f}')

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, [('Logistic Regression', lr), ('Random Forest', rf)]):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, lw=2, color='#2E75B6', label=f'AUC = {auc:.3f}')
    ax.plot([0,1],[0,1],'k--')
    ax.set_title(f'ROC — {name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feature_names = [f'V{i}' for i in range(1,29)] + ['scaled_amount','scaled_time']
importances   = rf.feature_importances_
indices       = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(12, 5))
plt.bar(range(15), importances[indices], color='steelblue', edgecolor='white')
plt.xticks(range(15), [feature_names[i] for i in indices], rotation=45, ha='right')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Importance Score')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Test a Single Prediction

In [ ]:
from predict import predict_single

sample_transaction = {
    'Time': 100, 'Amount': 149.62,
    'V1':-1.36,'V2':-0.07,'V3':2.54,'V4':1.38,'V5':-0.34,
    'V6':0.46,'V7':0.24,'V8':0.10,'V9':0.36,'V10':0.09,
    'V11':-0.55,'V12':-0.62,'V13':-0.99,'V14':-0.31,'V15':1.47,
    'V16':-0.47,'V17':0.21,'V18':0.03,'V19':0.40,'V20':0.25,
    'V21':-0.02,'V22':0.28,'V23':-0.11,'V24':0.07,'V25':0.13,
    'V26':-0.19,'V27':0.13,'V28':-0.02,
}

result = predict_single(sample_transaction, model_type='random_forest')
print('Prediction Result:')
for k, v in result.items():
    print(f'  {k}: {v}')